# 43 — Customer Lifecycle Orchestrator

A lifecycle-stage state machine that routes a customer account through `lead → onboarding → healthy → at_risk → renewal` stages, dispatching a different specialist agent at each stage while accumulating updates in a single typed `CustomerRecord`. A sixth Transition Agent inspects each specialist's output and decides — based on explicit rules — whether the account should advance, regress, or stay put.

## Framework Comparison

The lifecycle-stage state machine pattern used here — a stateful record, one specialist per stage, explicit transition rules — maps naturally to several popular agent frameworks:

| Framework | Equivalent construct | Key difference vs this example |
|-----------|---------------------|--------------------------------|
| **LangGraph** | `StateGraph` with typed `State`, conditional edges via `add_conditional_edges`, one node per lifecycle stage | LangGraph manages the graph execution loop and checkpointing automatically; here we own the loop explicitly in `workflow.py` |
| **LangChain** | `ConversationBufferMemory` + `RouterChain` (or `LLMRouterChain`) selecting the right specialist chain per stage | LangChain's router selects a chain at inference time; our Transition Agent uses LLM reasoning over explicit rules to decide whether to change stage |
| **CrewAI** | Conditional crew with `Process.sequential`, shared `crew.memory`, and a manager agent that gates which specialist `Agent` runs next | CrewAI crews run agents collaboratively with shared context; our orchestrator strictly serialises — one specialist per signal, no parallel agent activity |

In [ ]:
!pip install openai pydantic python-dotenv

In [ ]:
import os

# Replace the value below with your real key, or set it via the environment before running
os.environ["OPENAI_API_KEY"] = "sk-..."  # TODO: replace with your key

In [ ]:
# ruff: noqa: E402
"""
Pydantic models for the customer lifecycle orchestrator.

Covers the five lifecycle stages, incoming customer signals, the typed
CustomerRecord that accumulates state across invocations, per-stage
specialist outputs, and the top-level OrchestratorResult returned by run().
"""
from typing import Literal

from pydantic import BaseModel, Field

# ---------------------------------------------------------------------------
# Stage & signal enumerations
# ---------------------------------------------------------------------------

LifecycleStage = Literal["lead", "onboarding", "healthy", "at_risk", "renewal"]


class CustomerSignal(BaseModel):
    signal_type: Literal[
        "form_submit",
        "support_ticket",
        "nps_score",
        "login_activity",
        "contract_expiry",
        "churn_indicator",
        "upsell_interest",
    ] = Field(description="Category of the incoming signal.")
    value: str = Field(description="Signal payload as a string (e.g. NPS score, ticket ID, activity count).")
    timestamp: str = Field(description="ISO-8601 timestamp when the signal was recorded.")


# ---------------------------------------------------------------------------
# Core customer record
# ---------------------------------------------------------------------------


class CustomerRecord(BaseModel):
    customer_id: str = Field(description="Unique identifier for the customer account.")
    company_name: str = Field(description="Name of the customer's company.")
    stage: LifecycleStage = Field(description="Current lifecycle stage of the account.")
    health_score: float = Field(
        description="Composite health score between 0.0 (critical) and 1.0 (excellent).",
        ge=0.0,
        le=1.0,
    )
    arr_usd: float = Field(description="Annual recurring revenue in USD.")
    days_since_last_active: int = Field(
        description="Number of days since the account last logged in or used the product."
    )
    open_tickets: int = Field(description="Number of open support tickets for this account.")
    nps_score: int | None = Field(
        default=None, description="Most recent NPS score (0-10), or None if not yet collected."
    )
    notes: list[str] = Field(
        description="Chronological notes from agents or CSMs appended over the account lifetime."
    )
    signals: list[CustomerSignal] = Field(
        description="All signals received for this account in chronological order."
    )


# ---------------------------------------------------------------------------
# Per-stage specialist outputs
# ---------------------------------------------------------------------------


class QualificationResult(BaseModel):
    qualified: bool = Field(
        description="True if the lead meets the ICP threshold and should proceed to onboarding."
    )
    icp_score: float = Field(
        description="ICP fit score between 0.0 (poor fit) and 1.0 (perfect fit).",
        ge=0.0,
        le=1.0,
    )
    reasoning: str = Field(description="One to two sentence explanation of the qualification decision.")
    recommended_next_step: str = Field(
        description="Specific recommended action for the sales or CS team."
    )


class OnboardingStatus(BaseModel):
    tasks_complete: list[str] = Field(
        description="Onboarding tasks already completed or confirmed by the customer."
    )
    tasks_pending: list[str] = Field(description="Onboarding tasks still outstanding.")
    day1_ready: bool = Field(
        description="True if the customer has all prerequisites in place for a successful go-live."
    )
    blockers: list[str] = Field(description="Issues preventing forward progress in onboarding.")


class HealthAssessment(BaseModel):
    health_score: float = Field(
        description="Updated health score between 0.0 and 1.0 based on the latest signals.",
        ge=0.0,
        le=1.0,
    )
    risk_factors: list[str] = Field(
        description="Signals or patterns that are dragging health down."
    )
    positive_signals: list[str] = Field(
        description="Signals or patterns that are supporting a healthy account."
    )
    recommended_action: str = Field(
        description="Specific next action for the CSM or account team."
    )


class ChurnResponse(BaseModel):
    segment: Literal["escalate", "retain", "neutral"] = Field(
        description=(
            "'escalate' for accounts with active churn intent; "
            "'retain' for recoverable at-risk accounts; "
            "'neutral' for ambiguous signals."
        )
    )
    follow_up_draft: str = Field(
        description="Draft outreach message tailored to the segment and account context."
    )
    urgency: Literal["immediate", "this_week", "low"] = Field(
        description="Recommended response urgency for the account team."
    )


class RenewalPackage(BaseModel):
    renewal_probability: float = Field(
        description="Estimated probability (0.0-1.0) that the customer will renew.",
        ge=0.0,
        le=1.0,
    )
    negotiation_levers: list[str] = Field(
        description="Levers available to the CS or sales team to improve renewal odds."
    )
    outreach_draft: str = Field(
        description="Draft renewal outreach email or call script."
    )
    recommended_discount_pct: float = Field(
        description="Recommended discount percentage to offer (0.0 if none recommended).",
        ge=0.0,
        le=100.0,
    )


# ---------------------------------------------------------------------------
# Stage transition & orchestrator envelope
# ---------------------------------------------------------------------------


class StageOutput(BaseModel):
    stage_handled: LifecycleStage = Field(
        description="The stage whose specialist agent produced this output."
    )
    output: dict = Field(description="Raw dict of the specialist agent's structured response.")
    next_stage: LifecycleStage | None = Field(
        default=None,
        description="Stage to transition to, or None if the account remains in the current stage.",
    )
    updated_record: CustomerRecord = Field(
        description="CustomerRecord after applying this stage's updates."
    )


class OrchestratorResult(BaseModel):
    customer_id: str = Field(
        description="Unique identifier of the customer account that was processed."
    )
    previous_stage: LifecycleStage = Field(
        description="Lifecycle stage the account was in before this invocation."
    )
    stage_output: StageOutput = Field(
        description="Output from the specialist agent for the stage that was handled."
    )
    transition_occurred: bool = Field(
        description="True if the orchestrator moved the account to a new lifecycle stage."
    )

In [ ]:
"""
System prompt constants for each lifecycle-stage specialist agent
and the transition-decision agent.

Each constant is a plain string passed as the 'system' role message
to the OpenAI chat completions API.
"""

# ---------------------------------------------------------------------------
# Stage: lead
# ---------------------------------------------------------------------------

QUALIFY_SYSTEM = (
    "You are a sales qualification analyst. Given a CustomerRecord for a lead account "
    "and their latest signals, score the lead against this Ideal Customer Profile (ICP):\n\n"
    "  Industry:     SaaS, FinTech, or E-commerce\n"
    "  Company size: 50-500 employees implied by ARR range $50k-$2M\n"
    "  Pain point:   manual workflows, data silos, compliance burden, or growth bottlenecks\n"
    "  Buyer signal: form submission or demo request with specific use-case mentioned\n"
    "  Budget signal: ARR > $0 or explicit budget mentioned in notes/signals\n\n"
    "Scoring:\n"
    "  0.8-1.0 → qualified (3+ ICP criteria met, clear pain and budget signal)\n"
    "  0.5-0.79 → borderline (2 criteria met, escalate to human review)\n"
    "  0.0-0.49 → not qualified (fewer than 2 criteria met)\n\n"
    "Set qualified=true only if icp_score >= 0.7. "
    "recommended_next_step must be a concrete action (e.g. 'Schedule discovery call within 24h'). "
    "Never invent data not present in the record or signals. "
    "Return a QualificationResult JSON object."
)

# ---------------------------------------------------------------------------
# Stage: onboarding
# ---------------------------------------------------------------------------

ONBOARD_SYSTEM = (
    "You are a customer onboarding specialist. Given a CustomerRecord in the onboarding stage "
    "and their latest signals, assess how far along the customer is and what remains.\n\n"
    "Standard onboarding checklist:\n"
    "  1. Welcome call completed\n"
    "  2. Admin users provisioned\n"
    "  3. SSO or authentication configured\n"
    "  4. Data migration or import completed\n"
    "  5. Core workflow configured and tested\n"
    "  6. Team training session scheduled\n"
    "  7. Go-live date confirmed\n\n"
    "Infer task status from signals, notes, days_since_last_active, and open_tickets. "
    "tasks_complete should list items confirmed done. tasks_pending lists items still outstanding. "
    "day1_ready is true only when all critical path tasks (items 1-5) are in tasks_complete. "
    "blockers lists any specific issues preventing progress. "
    "Return an OnboardingStatus JSON object."
)

# ---------------------------------------------------------------------------
# Stage: healthy
# ---------------------------------------------------------------------------

HEALTH_SYSTEM = (
    "You are a customer health analyst. Given a CustomerRecord in the healthy stage "
    "and their latest signals, produce an updated health assessment.\n\n"
    "Health score guidance:\n"
    "  0.8-1.0 → excellent: high engagement, low tickets, positive NPS, recent activity\n"
    "  0.6-0.79 → good: mostly positive but one or two mild concerns\n"
    "  0.4-0.59 → fair: mixed signals; proactive intervention recommended\n"
    "  0.0-0.39 → poor: multiple risk factors; account may need escalation to at_risk\n\n"
    "risk_factors lists specific negative signals (e.g. '14 days inactive', '3 open tickets'). "
    "positive_signals lists specific positive signals (e.g. 'NPS 9', 'upsell interest signal'). "
    "recommended_action must be a single concrete next step for the CSM. "
    "Base the health_score on the combined weight of signals, not just one metric. "
    "Return a HealthAssessment JSON object."
)

# ---------------------------------------------------------------------------
# Stage: at_risk
# ---------------------------------------------------------------------------

CHURN_SYSTEM = (
    "You are a customer retention specialist. Given a CustomerRecord in the at_risk stage "
    "and their latest signals, classify the account and draft a personalised outreach.\n\n"
    "Segments:\n"
    "  'escalate': Active churn intent — customer has signalled they are leaving, "
    "submitted a cancellation request, or NPS is 0-4 with strong negative language. "
    "Draft urgent outreach from a senior executive within 24 hours.\n"
    "  'retain': Recoverable at-risk — declining engagement or low NPS (5-6) but no "
    "explicit churn signal. Draft a warm, value-focused outreach from the CSM.\n"
    "  'neutral': Ambiguous signals — minor dip in activity or a single negative ticket. "
    "Draft a light check-in that surfaces any unmet needs.\n\n"
    "urgency must match the segment: escalate=immediate, retain=this_week, neutral=low. "
    "follow_up_draft should be 3-5 sentences, personalised to company_name and specific signals. "
    "Return a ChurnResponse JSON object."
)

# ---------------------------------------------------------------------------
# Stage: renewal
# ---------------------------------------------------------------------------

RENEWAL_SYSTEM = (
    "You are a renewal and expansion specialist. Given a CustomerRecord in the renewal stage "
    "and their latest signals, build a renewal package.\n\n"
    "renewal_probability guidance:\n"
    "  health_score >= 0.7 and NPS >= 8 → high probability 0.75-0.95\n"
    "  health_score 0.5-0.69 or NPS 6-7 → medium probability 0.45-0.74\n"
    "  health_score < 0.5 or NPS < 6 → low probability 0.1-0.44\n\n"
    "negotiation_levers: list 3-5 specific levers (e.g. 'multi-year discount', "
    "'add seats bundle', 'executive sponsor alignment', 'case study offer'). "
    "outreach_draft: 4-6 sentence personalised renewal email referencing company_name, "
    "arr_usd, and the contract_expiry signal if present. "
    "recommended_discount_pct: 0.0 if health is strong; up to 15.0 for at-risk renewals. "
    "Return a RenewalPackage JSON object."
)

# ---------------------------------------------------------------------------
# Transition agent
# ---------------------------------------------------------------------------

TRANSITION_SYSTEM = (
    "You are a customer lifecycle stage manager. Given a CustomerRecord (current stage and "
    "full context) and the output dict from the stage's specialist agent, decide whether "
    "the account should transition to a different lifecycle stage.\n\n"
    "Transition rules:\n"
    "  lead → onboarding:  qualified=true in QualificationResult\n"
    "  lead → lead:        qualified=false — stay, nurture\n"
    "  onboarding → healthy: day1_ready=true and tasks_pending is empty or near-empty\n"
    "  onboarding → at_risk: open_tickets > 3 or days_since_last_active > 14 during onboarding\n"
    "  healthy → at_risk:  health_score < 0.5 in HealthAssessment\n"
    "  healthy → renewal:  contract_expiry signal present and health_score >= 0.6\n"
    "  at_risk → healthy:  ChurnResponse segment='retain' and health_score >= 0.6\n"
    "  at_risk → renewal:  contract_expiry signal present regardless of health\n"
    "  renewal → healthy:  renewal_probability >= 0.7 (after renewal closed)\n"
    "  renewal → at_risk:  renewal_probability < 0.4\n\n"
    "Return a JSON object with exactly three fields:\n"
    "  should_transition (bool): whether to move to a new stage\n"
    "  next_stage (string or null): the target stage, or null if no transition\n"
    "  reasoning (string): one sentence explaining the decision\n"
)

In [ ]:
# ruff: noqa: E402
"""
Lifecycle-stage state machine orchestrator.

run() is the single public entrypoint. It:
  1. Appends the incoming signal to the CustomerRecord.
  2. Dispatches to the specialist agent for the CURRENT stage only.
  3. Asks the transition agent whether a stage change is warranted.
  4. Updates the record and returns an OrchestratorResult.
"""
import json
from typing import Any

from openai import OpenAI

_client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])
_MODEL = "gpt-4o-mini"

# Map each stage to its specialist (system prompt, schema name, Pydantic model)
_STAGE_AGENTS: dict[str, tuple[str, str, Any]] = {
    "lead": (QUALIFY_SYSTEM, "QualificationResult", QualificationResult),
    "onboarding": (ONBOARD_SYSTEM, "OnboardingStatus", OnboardingStatus),
    "healthy": (HEALTH_SYSTEM, "HealthAssessment", HealthAssessment),
    "at_risk": (CHURN_SYSTEM, "ChurnResponse", ChurnResponse),
    "renewal": (RENEWAL_SYSTEM, "RenewalPackage", RenewalPackage),
}


def _run_stage_agent(
    record: CustomerRecord,
    system: str,
    schema_name: str,
    schema: Any,
) -> dict:
    """Call the specialist agent for the current stage and return its output as a dict."""
    resp = _client.chat.completions.create(
        model=_MODEL,
        messages=[
            {"role": "system", "content": system},
            {"role": "user", "content": record.model_dump_json()},
        ],
        response_format={
            "type": "json_schema",
            "json_schema": {
                "name": schema_name,
                "strict": True,
                "schema": schema.model_json_schema(),
            },
        },
    )
    validated = schema.model_validate_json(resp.choices[0].message.content)
    return validated.model_dump()


def _check_transition(
    record: CustomerRecord,
    stage_output_dict: dict,
) -> tuple[bool, LifecycleStage | None]:
    """
    Ask the transition agent whether the account should move to a new stage.

    Returns (should_transition, next_stage).
    """
    payload = {
        "customer_record": record.model_dump(),
        "stage_output": stage_output_dict,
    }
    resp = _client.chat.completions.create(
        model=_MODEL,
        messages=[
            {"role": "system", "content": TRANSITION_SYSTEM},
            {"role": "user", "content": json.dumps(payload)},
        ],
        response_format={"type": "json_object"},
    )
    decision = json.loads(resp.choices[0].message.content)
    should_transition: bool = bool(decision.get("should_transition", False))
    next_stage_raw: str | None = decision.get("next_stage")
    next_stage: LifecycleStage | None = (
        next_stage_raw if should_transition and next_stage_raw else None
    )
    return should_transition, next_stage


def _update_record(
    record: CustomerRecord,
    stage_output_dict: dict,
    next_stage: LifecycleStage | None,
) -> CustomerRecord:
    """
    Apply updates from the stage output back to the CustomerRecord.

    Updates health_score if the output contains one, transitions the stage
    if next_stage is set, and appends a summary note.
    """
    updates: dict = {}

    if next_stage is not None:
        updates["stage"] = next_stage

    # Propagate health_score from HealthAssessment or RenewalPackage outputs
    if "health_score" in stage_output_dict:
        updates["health_score"] = stage_output_dict["health_score"]

    # Propagate NPS if a new nps_score signal was submitted
    if "nps_score" in stage_output_dict:
        updates["nps_score"] = stage_output_dict["nps_score"]

    # Append a brief agent note for traceability
    current_stage = record.stage
    note = f"[{current_stage}] agent ran"
    if next_stage:
        note += f"; transitioned to {next_stage}"
    updated_notes = list(record.notes) + [note]
    updates["notes"] = updated_notes

    return record.model_copy(update=updates)


def run(record: CustomerRecord, signal: CustomerSignal) -> OrchestratorResult:
    """
    Process a single incoming signal for a customer account.

    Steps:
      1. Append the signal to the record.
      2. Dispatch to the specialist agent for the current stage.
      3. Check whether a stage transition is warranted.
      4. Update the record with any changes.
      5. Return an OrchestratorResult.
    """
    previous_stage: LifecycleStage = record.stage

    # 1. Append signal
    updated_signals = list(record.signals) + [signal]
    record = record.model_copy(update={"signals": updated_signals})

    # 2. Dispatch to the correct specialist agent
    system, schema_name, schema = _STAGE_AGENTS[record.stage]
    stage_output_dict = _run_stage_agent(record, system, schema_name, schema)

    # 3. Check transition
    should_transition, next_stage = _check_transition(record, stage_output_dict)

    # 4. Update record
    updated_record = _update_record(record, stage_output_dict, next_stage)

    return OrchestratorResult(
        customer_id=record.customer_id,
        previous_stage=previous_stage,
        stage_output=StageOutput(
            stage_handled=previous_stage,
            output=stage_output_dict,
            next_stage=next_stage,
            updated_record=updated_record,
        ),
        transition_occurred=should_transition and next_stage is not None,
    )

In [ ]:
# ------------------------------------------------------------------
# Scenario 1 — Customer A: lead who just submitted an inbound form
# ------------------------------------------------------------------

def _print_result(label: str, result) -> None:
    print(f"\n{'=' * 60}")
    print(f"  {label}")
    print(f"{'=' * 60}")
    print(f"  Customer : {result.customer_id}")
    print(f"  Stage    : {result.previous_stage}  ->  ", end="")
    if result.transition_occurred:
        print(result.stage_output.next_stage)
    else:
        print(f"{result.previous_stage} (no change)")
    print(f"  Transition: {result.transition_occurred}")
    print("\n  Stage output (summary):")
    for k, v in result.stage_output.output.items():
        display = str(v)
        if len(display) > 120:
            display = display[:117] + "..."
        print(f"    {k}: {display}")
    print()


customer_a = CustomerRecord(
    customer_id="CUS-001",
    company_name="Flowstate Analytics",
    stage="lead",
    health_score=0.5,
    arr_usd=0.0,
    days_since_last_active=0,
    open_tickets=0,
    nps_score=None,
    notes=["Inbound demo request from CFO via website form."],
    signals=[],
)
signal_a = CustomerSignal(
    signal_type="form_submit",
    value=(
        "Demo request: 'We need to automate our compliance reporting"
        " -- current process is 100% manual.'"
    ),
    timestamp="2026-06-19T08:15:00Z",
)

result_a = run(customer_a, signal_a)
_print_result("Customer A -- Lead Qualification", result_a)

In [ ]:
# ------------------------------------------------------------------
# Scenario 2 — Customer B: at-risk customer with churn indicator
# ------------------------------------------------------------------

customer_b = CustomerRecord(
    customer_id="CUS-002",
    company_name="Meridian Logistics",
    stage="at_risk",
    health_score=0.31,
    arr_usd=84000.0,
    days_since_last_active=18,
    open_tickets=6,
    nps_score=3,
    notes=[
        "Health dropped after Q1 migration -- data sync issues.",
        "CSM flagged account for executive attention on 2026-05-10.",
    ],
    signals=[
        CustomerSignal(
            signal_type="support_ticket",
            value="TKT-4421: Bulk export broken since 2.1 upgrade",
            timestamp="2026-06-01T11:00:00Z",
        ),
        CustomerSignal(
            signal_type="support_ticket",
            value="TKT-4502: Dashboard loading errors -- 504 timeouts",
            timestamp="2026-06-10T14:30:00Z",
        ),
    ],
)
signal_b = CustomerSignal(
    signal_type="churn_indicator",
    value="Customer emailed: 'We are evaluating alternatives and may not renew in August.'",
    timestamp="2026-06-19T09:00:00Z",
)

result_b = run(customer_b, signal_b)
_print_result("Customer B -- At-Risk Churn Response", result_b)

## Starter Exercise

**Add an `upsell` stage between `healthy` and `renewal`.**

When a `healthy` customer sends an `upsell_interest` signal, the orchestrator should transition to `upsell` and dispatch a new `UpsellProposal` agent that recommends a tier upgrade and drafts a commercial outreach.

**What schema models, prompts, and transition rules would you add?**

Think through:
- What fields does a `UpsellProposal` model need? (e.g. recommended tier, price delta, business case, outreach draft)
- What should the `UPSELL_SYSTEM` prompt instruct the agent to look for in the `CustomerRecord`?
- Which existing transition rules need to change? What new rules enter `upsell` and exit it?
- How does `_STAGE_AGENTS` need to be updated?
- Does `LifecycleStage` need a new literal?

Try writing the answer yourself before looking at the answer key below.

In [ ]:
# ---------------------------------------------------------------------------
# Answer key — UpsellProposal schema model
# ---------------------------------------------------------------------------

class UpsellProposal(BaseModel):
    recommended_tier: str = Field(
        description="Name of the tier or plan being recommended (e.g. 'Enterprise', 'Pro Plus')."
    )
    current_arr_usd: float = Field(
        description="Customer's current ARR in USD, taken directly from the CustomerRecord."
    )
    proposed_arr_usd: float = Field(
        description="Projected ARR in USD after the tier upgrade."
    )
    business_case: str = Field(
        description="Two to three sentence rationale explaining the value the customer gains from upgrading."
    )
    outreach_draft: str = Field(
        description="Draft commercial outreach email or call script personalised to the customer's company and signals."
    )
    confidence: float = Field(
        description="Confidence score (0.0-1.0) that the customer is ready to upgrade based on current signals.",
        ge=0.0,
        le=1.0,
    )


# ---------------------------------------------------------------------------
# Answer key — UPSELL_SYSTEM prompt constant
# ---------------------------------------------------------------------------

UPSELL_SYSTEM = (
    "You are a commercial expansion specialist. Given a CustomerRecord in the upsell stage "
    "and their latest signals (especially an upsell_interest signal), recommend a tier upgrade "
    "and draft a commercial outreach message.\n\n"
    "Guidance:\n"
    "  - recommended_tier: choose the next natural tier above the customer's current ARR band.\n"
    "  - proposed_arr_usd: estimate new ARR based on typical tier pricing; do not invent numbers "
    "not inferable from the record.\n"
    "  - business_case: anchor to specific pain points or positive signals in the record "
    "(e.g. high seat usage, upsell_interest signal value, NPS score).\n"
    "  - outreach_draft: 4-5 sentences, personalised to company_name and current context; "
    "reference the upsell_interest signal payload directly.\n"
    "  - confidence: high (0.75-1.0) if health_score >= 0.7 and a clear upsell_interest signal "
    "is present; medium (0.45-0.74) if health is good but interest is inferred; "
    "low (0.0-0.44) if signals are ambiguous.\n"
    "Return a UpsellProposal JSON object."
)


# ---------------------------------------------------------------------------
# Answer key — updated TRANSITION_SYSTEM rules mentioning the new stage
# ---------------------------------------------------------------------------

# The TRANSITION_SYSTEM string would gain two new rules in the Transition rules block:
#
#   healthy → upsell:   upsell_interest signal present and health_score >= 0.7
#   upsell  → renewal:  contract_expiry signal present or confidence >= 0.75
#   upsell  → healthy:  confidence < 0.45 (customer not yet ready; return to healthy nurture)
#
# And LifecycleStage would expand to:
#
#   LifecycleStage = Literal["lead", "onboarding", "healthy", "upsell", "at_risk", "renewal"]
#
# And _STAGE_AGENTS would gain:
#
#   "upsell": (UPSELL_SYSTEM, "UpsellProposal", UpsellProposal),

print("UpsellProposal schema fields:", list(UpsellProposal.model_fields.keys()))
print()
print("UPSELL_SYSTEM (first 200 chars):", UPSELL_SYSTEM[:200])
print()
print("New LifecycleStage literal:")
print('  Literal["lead", "onboarding", "healthy", "upsell", "at_risk", "renewal"]')
print()
print("New _STAGE_AGENTS entry:")
print('  "upsell": (UPSELL_SYSTEM, "UpsellProposal", UpsellProposal)')
print()
print("New transition rules:")
print("  healthy → upsell:  upsell_interest signal present and health_score >= 0.7")
print("  upsell  → renewal: contract_expiry signal present or confidence >= 0.75")
print("  upsell  → healthy: confidence < 0.45")